## RQ3: Table 10 - Per-application distinct recall

In [1]:
import json
import os
import glob
import pandas as pd
from collections import defaultdict
import re

In [2]:
import sys
import os
# Add the parent directory (RQs) to the path to find the config file
if '..' not in sys.path:
    sys.path.insert(1, os.path.abspath('..'))
import config

In [3]:
def canonicalize(pii, pii_type):
    """Applies canonicalization rules to a PII string based on its type."""
    if not isinstance(pii, str):
        pii = str(pii)
    
    pii = pii.strip()
    
    if pii_type == 'EMAIL':
        return pii.lower()
    elif pii_type == 'PHONE':
        is_plus_prefix = pii.startswith('+')
        digits = re.sub(r'\D', '', pii)
        return ('+' + digits) if is_plus_prefix else digits
    elif pii_type == 'PERSON_NAME':
        return pii.lower()
    
    return pii # Default for USERNAME, POSTAL_ADDRESS etc. is just stripping whitespace

def parse_filename_app_id(filepath):
    """Parses a filename to extract just the app ID."""
    base_name = os.path.basename(filepath)
    match = re.match(r'PII_([A-Z0-9]+)_', base_name)
    if match:
        return match.group(1)
    return None

def load_pii_data(path):
    """Loads and aggregates all distinct canonicalized PII from a directory."""
    # Structure: {app_id: {pii_type: {set of pii_strings}}}
    data = defaultdict(lambda: defaultdict(set))
    files = glob.glob(os.path.join(path, '*.jsonl'))
    
    for f_path in files:
        app_id = parse_filename_app_id(f_path)
        if not app_id:
            continue
        with open(f_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    record = json.loads(line)
                    pii_type = record['PII_type']
                    if pii_type in config.PII_TYPES:
                        for pii_item in record.get('PII', []):
                            canon_pii = canonicalize(pii_item, pii_type)
                            if canon_pii:
                                data[app_id][pii_type].add(canon_pii)
                except json.JSONDecodeError:
                    print(f"Warning: Could not decode JSON from line in {f_path}")
    return data

gt_pii_sets = load_pii_data(os.path.join('..', config.GROUND_TRUTH_DIR))
system_pii_sets = load_pii_data(os.path.join('..', config.GPT4O_RESULTS_DIR))

In [4]:
table_data = []

for app_id, app_name in config.APP_MAPPING.items():
    row = {'ID': app_id, 'Application': app_name}
    
    all_gt_pii_for_app = set()
    all_system_pii_for_app = set()

    # --- Per-PII Type Recall Calculation ---
    for pii_type in config.PII_TYPES:
        col_name = config.COLUMN_MAPPING[pii_type]
        
        gt_set = gt_pii_sets.get(app_id, {}).get(pii_type, set())
        system_set = system_pii_sets.get(app_id, {}).get(pii_type, set())
        
        all_gt_pii_for_app.update(gt_set)
        all_system_pii_for_app.update(system_set)
        
        if not gt_set:
            row[col_name] = '-'
        else:
            # Ra,t = |Ga,t ∩ Sa,t| / |Ga,t|
            recall = len(gt_set.intersection(system_set)) / len(gt_set)
            row[col_name] = recall

    # --- All PII Recall Calculation ---
    if not all_gt_pii_for_app:
        row['All PII'] = '-'
    else:
        # Ra,all = |(U Ga,t) ∩ (U Sa,t)| / |U Ga,t|
        all_recall = len(all_gt_pii_for_app.intersection(all_system_pii_for_app)) / len(all_gt_pii_for_app)
        row['All PII'] = all_recall
        
    table_data.append(row)

In [5]:
df = pd.DataFrame(table_data)

# Reorder columns to match Table 10
final_columns = ['ID', 'Application'] + [config.COLUMN_MAPPING[pt] for pt in config.PII_TYPES] + ['All PII']
df = df[final_columns]

df = df.set_index('ID')

# Format numbers to 2 decimal places, replacing non-numeric placeholders with '-‘
for col in df.columns:
    if col != 'Application':
        df[col] = df[col].apply(lambda x: f"{x:.2f}" if isinstance(x, float) else x)

# Display the dataframe
df

,Application,Email,Phone,User Name,Person Name,Postal Address,All PII
ID,,,,,,,
A1,WhatsApp,-,0.96,0.50,0.68,-,0.79
A2,Snapchat,1.00,1.00,0.33,1.00,-,0.79
A3,Telegram,-,-,-,-,-,-
A4,Google Maps,1.00,-,1.00,-,-,1.00
A5,Samsung Internet,1.00,-,0.00,-,-,0.20
I1,WhatsApp (iOS),-,-,-,1.00,1.00,1.00
I2,Contacts,1.00,0.47,-,0.86,-,0.65
I3,Apple Messages,1.00,0.00,0.00,1.00,-,0.33
I4,Safari,-,-,0.02,-,-,0.02


In [6]:
# Optional: Save to LaTeX
latex_output = df.to_latex(index=True, caption='Per-application distinct recall.', label='tab:app_level_recall', na_rep='-')
print(latex_output)

\begin{table}
\caption{Per-application distinct recall.}
\label{tab:app_level_recall}
\begin{tabular}{llllllll}
\toprule
 & Application & Email & Phone & User Name & Person Name & Postal Address & All PII \\
ID &  &  &  &  &  &  &  \\
\midrule
A1 & WhatsApp & - & 0.96 & 0.50 & 0.68 & - & 0.79 \\
A2 & Snapchat & 1.00 & 1.00 & 0.33 & 1.00 & - & 0.79 \\
A3 & Telegram & - & - & - & - & - & - \\
A4 & Google Maps & 1.00 & - & 1.00 & - & - & 1.00 \\
A5 & Samsung Internet & 1.00 & - & 0.00 & - & - & 0.20 \\
I1 & WhatsApp (iOS) & - & - & - & 1.00 & 1.00 & 1.00 \\
I2 & Contacts & 1.00 & 0.47 & - & 0.86 & - & 0.65 \\
I3 & Apple Messages & 1.00 & 0.00 & 0.00 & 1.00 & - & 0.33 \\
I4 & Safari & - & - & 0.02 & - & - & 0.02 \\
I5 & Calendar & 1.00 & - & - & - & - & 1.00 \\
\bottomrule
\end{tabular}
\end{table}

